# Post-Processing Toolbox Demo
Independent implementation of the post processing toolbox without networking constraints.

In [1]:
import random
import asyncio
from typing import List
from PostProcessingToolbox import (
    CascadeClientProtocol,
    WinnowClientProtocol,
    PolarClientProtocol,
    NR_LDPC_Standard_ClientProtocol,
    LDPC_RateAdaptive_ClientProtocol,
    PrivacyAmplification,
    ParityOracle
)

class LocalMockOracle(ParityOracle):
    def __init__(self, key: List[int]):
        self.key = key
    async def get_parity(self, indices: List[int]) -> int:
        p_val = 0
        for idx in indices:
            p_val ^= self.key[idx]
        return p_val
    async def get_parities(self, blocks: List[List[int]]) -> List[int]:
        return [await self.get_parity(block) for block in blocks]

def generate_keys(N: int, qber: float):
    alice_key = [random.randint(0, 1) for _ in range(N)]
    bob_key = list(alice_key)
    for i in range(N):
        if random.random() < qber:
            bob_key[i] = 1 - bob_key[i]
    return alice_key, bob_key

[NR-LDPC] GPU Detected: 1 device(s). Acceleration ENABLED.
[NR-LDPC] Sionna Library Loaded Successfully.


### 1. Cascade Demo

In [2]:
async def demo_cascade():
    N, qber = 1000, 0.04
    print(f"\n{'='*50}\n--- 1. Cascade Error Correction Demo ---\n{'='*50}")
    print(f"[*] Alice generates {N} bits of reference key.")
    print(f"[*] Bob receives key over noisy channel with ~{qber:.1%} probability of error per bit.")
    alice_key, bob_key = generate_keys(N, qber)
    initial_errors = sum(1 for a, b in zip(alice_key, bob_key) if a != b)
    print(f"[*] Actual errors introduced: {initial_errors} ({(initial_errors/N):.2%} QBER)\n")
    
    print("[*] Setting up Local Oracle (Alice) and Cascade Client (Bob)...")
    cascade = CascadeClientProtocol(num_passes=4, verbose=True)
    
    print("[*] Bob starts Cascade algorithm to reconcile keys with Alice's Oracle in multiple passes...")
    corrected, revealed, total_corrected, channel_uses = await cascade.run(bob_key, qber, LocalMockOracle(alice_key))
    
    print("\n[*] Cascade algorithm completed. Verifying results...")
    final_errors = sum(1 for a, b in zip(alice_key, corrected) if a != b)
    print(f"\n--- Cascade Summary ---")
    print(f"[*] Initial Errors: {initial_errors}")
    print(f"[*] Errors Remaining: {final_errors}")
    print(f"[*] Parity Bits Revealed (Leakage): {revealed}")
    print(f"[*] Communication Rounds (Channel Uses): {channel_uses}")
    if final_errors == 0:
        print("[+] Cascade Match Success! Bob's key perfectly matches Alice's key.\n")
    else:
        print("[-] Cascade Match Failed! Some errors remain.\n")
        
    return alice_key, corrected

alice_key, matched_key = await demo_cascade()


--- 1. Cascade Error Correction Demo ---
[*] Alice generates 1000 bits of reference key.
[*] Bob receives key over noisy channel with ~4.0% probability of error per bit.
[*] Actual errors introduced: 36 (3.60% QBER)

[*] Setting up Local Oracle (Alice) and Cascade Client (Bob)...
[*] Bob starts Cascade algorithm to reconcile keys with Alice's Oracle in multiple passes...
Starting Cascade Protocol (Client) with 4 passes.
Initial Key (First 50 bits): 01000111000101110011101010000011010001101000111111...

--- Pass 1 ---
  Block size k: 18
  Blocks with odd parity errors: 20 / 56
  Correcting error in block 0...
    [BINARY] Mismatch in subset (9 bits). Searching left.
    [BINARY] Match in subset. Error in right half (4 bits). Searching right.
    [BINARY] Match in subset. Error in right half (2 bits). Searching right.
    [BINARY] Mismatch in subset (1 bits). Searching left.
  [FIXED] Found error at index 7. Flipping bit 1 -> 0
  Correcting error in block 2...
    [BINARY] Mismatch in s

### 2. Winnow Demo

In [3]:
async def demo_winnow():
    N, qber = 1000, 0.04
    print(f"\n{'='*50}\n--- 2. Winnow Error Correction Demo ---\n{'='*50}")
    print(f"[*] Alice generates {N} bits of reference key.")
    print(f"[*] Bob receives key over noisy channel with ~{qber:.1%} probability of error per bit.")
    alice_key, bob_key = generate_keys(N, qber)
    initial_errors = sum(1 for a, b in zip(alice_key, bob_key) if a != b)
    print(f"[*] Actual errors introduced: {initial_errors} ({(initial_errors/N):.2%} QBER)\n")
    
    print("[*] Setting up Local Oracle (Alice) and Winnow Client (Bob)...")
    winnow = WinnowClientProtocol(num_passes=4, verbose=True)
    
    print("[*] Bob starts Winnow algorithm using Hamming codes to find and correct errors...")
    corrected, revealed, total_corrected, channel_uses = await winnow.run(bob_key, qber, LocalMockOracle(alice_key))
    
    print("\n[*] Winnow algorithm completed. Verifying results...")
    final_errors = sum(1 for a, b in zip(alice_key, corrected) if a != b)
    print(f"\n--- Winnow Summary ---")
    print(f"[*] Initial Errors: {initial_errors}")
    print(f"[*] Errors Remaining: {final_errors}")
    print(f"[*] Parity Bits Revealed (Leakage): {revealed}")
    print(f"[*] Communication Rounds (Channel Uses): {channel_uses}")
    if final_errors == 0:
        print("[+] Winnow Match Success! Bob's key perfectly matches Alice's key.\n")
    else:
        print("[-] Winnow Match Failed! Some errors remain.\n")

await demo_winnow()


--- 2. Winnow Error Correction Demo ---
[*] Alice generates 1000 bits of reference key.
[*] Bob receives key over noisy channel with ~4.0% probability of error per bit.
[*] Actual errors introduced: 34 (3.40% QBER)

[*] Setting up Local Oracle (Alice) and Winnow Client (Bob)...
[*] Bob starts Winnow algorithm using Hamming codes to find and correct errors...
Starting Winnow Protocol (Client) with 4 passes.
Initial Key (First 50 bits): 01100100101110001110110100100000110000011110100000...

--- Pass 1 ---
  Block size k: 8
  Blocks with odd parity errors: 26 / 125
    [WINNOW] Flipped bit at index 28 (block pos 4)
    [WINNOW] Flipped bit at index 71 (block pos 7)
    [WINNOW] Flipped bit at index 125 (block pos 5)
    [WINNOW] Flipped bit at index 132 (block pos 4)
    [WINNOW] Flipped bit at index 151 (block pos 7)
    [WINNOW] Flipped bit at index 177 (block pos 1)
    [WINNOW] Flipped bit at index 208 (block pos 0)
    [WINNOW] Flipped bit at index 301 (block pos 5)
    [WINNOW] Fli

### 3. Polar Codes Demo

In [4]:
async def demo_polar():
    N, qber = 1000, 0.03
    print(f"\n{'='*50}\n--- 3. Polar Codes Error Correction Demo ---\n{'='*50}")
    print(f"[*] Alice generates {N} bits of reference key.")
    print(f"[*] Bob receives key over noisy channel with ~{qber:.1%} probability of error per bit.")
    alice_key, bob_key = generate_keys(N, qber)
    initial_errors = sum(1 for a, b in zip(alice_key, bob_key) if a != b)
    print(f"[*] Actual errors introduced: {initial_errors} ({(initial_errors/N):.2%} QBER)\n")
    
    print("[*] Setting up Local Oracle (Alice) and Polar Client (Bob)...")
    polar = PolarClientProtocol(verbose=True, u_fer_target=0.01)
    
    print("[*] Bob executes Successive Cancellation decoding along the polar combinations...")
    corrected, revealed, total_corrected, channel_uses = await polar.run(bob_key, qber, LocalMockOracle(alice_key))
    
    print("\n[*] Polar decoding completed. Verifying results...")
    final_errors = sum(1 for a, b in zip(alice_key, corrected) if a != b)
    print(f"\n--- Polar Summary ---")
    print(f"[*] Initial Errors: {initial_errors}")
    print(f"[*] Errors Remaining: {final_errors}")
    print(f"[*] Bits Revealed via Oracle: {revealed}")
    if final_errors == 0:
        print("[+] Polar Match Success! Bob's key perfectly matches Alice's key.\n")
    else:
        print("[-] Polar Match Failed! Some errors remain.\n")

await demo_polar()


--- 3. Polar Codes Error Correction Demo ---
[*] Alice generates 1000 bits of reference key.
[*] Bob receives key over noisy channel with ~3.0% probability of error per bit.
[*] Actual errors introduced: 29 (2.90% QBER)

[*] Setting up Local Oracle (Alice) and Polar Client (Bob)...
[*] Bob executes Successive Cancellation decoding along the polar combinations...
Starting Polar Protocol (Client) with N=1024, target FER=0.01

[*] Polar decoding completed. Verifying results...

--- Polar Summary ---
[*] Initial Errors: 29
[*] Errors Remaining: 0
[*] Bits Revealed via Oracle: 305
[+] Polar Match Success! Bob's key perfectly matches Alice's key.



### 4. NR-LDPC Standard Demo

In [11]:
async def demo_nr_ldpc():
    N, qber = 1000, 0.1
    print(f"\n{'='*50}\n--- 4. NR-LDPC Standard Error Correction Demo ---\n{'='*50}")
    
    if NR_LDPC_Standard_ClientProtocol is None:
        print("[-] NR-LDPC Standard is NOT available (TensorFlow/Sionna not installed).")
        print("[*] This protocol requires: pip install tensorflow sionna")
        print("[*] Skipping NR-LDPC demo.\n")
        return None, None
    
    print(f"[*] Alice generates {N} bits of reference key.")
    print(f"[*] Bob receives key over noisy channel with ~{qber:.1%} probability of error per bit.")
    alice_key, bob_key = generate_keys(N, qber)
    initial_errors = sum(1 for a, b in zip(alice_key, bob_key) if a != b)
    print(f"[*] Actual errors introduced: {initial_errors} ({(initial_errors/N):.2%} QBER)\n")
    
    print("[*] Setting up Local Oracle (Alice) and NR-LDPC Standard Client (Bob)...")
    try:
        nr_ldpc = NR_LDPC_Standard_ClientProtocol(verbose=True, rate=0.333)
        print("[*] Bob executes 5G NR LDPC belief propagation to decode and fix errors...")
        corrected, revealed, total_corrected, channel_uses = await nr_ldpc.run(bob_key, qber, LocalMockOracle(alice_key))
        
        print("\n[*] NR-LDPC decoding completed. Verifying results...")
        final_errors = sum(1 for a, b in zip(alice_key, corrected) if a != b)
        print(f"\n--- NR-LDPC Summary ---")
        print(f"[*] Initial Errors: {initial_errors}")
        print(f"[*] Errors Remaining: {final_errors}")
        print(f"[*] Parity Bits Revealed (Leakage): {revealed}")
        print(f"[*] Communication Rounds (Channel Uses): {channel_uses}")
        if final_errors == 0:
            print("[+] NR-LDPC Match Success! Bob's key perfectly matches Alice's key.\n")
        else:
            print("[-] NR-LDPC Match Failed! Some errors remain.\n")
        return alice_key, corrected
    except Exception as e:
        print(f"\n[-] NR-LDPC Failed during execution: {e}\n")
        return None, None

nr_ldpc_keys = await demo_nr_ldpc()


--- 4. NR-LDPC Standard Error Correction Demo ---
[*] Alice generates 1000 bits of reference key.
[*] Bob receives key over noisy channel with ~10.0% probability of error per bit.
[*] Actual errors introduced: 117 (11.70% QBER)

[*] Setting up Local Oracle (Alice) and NR-LDPC Standard Client (Bob)...
[*] Bob executes 5G NR LDPC belief propagation to decode and fix errors...
[NR-LDPC-STD] Initializing 5G LDPC Engine (k=1000, Mother n=3000, Base Rate=0.333)...
[NR-LDPC-STD] Code Configured: n=3000 (Actual Rate 0.333)
[NR-LDPC-STD] Computing Generator Matrix G...
[NR-LDPC-STD] G Matrix Ready. Systematic Bits: 792
[NR-LDPC-STD] Rate Matching: QBER=0.1000 => Need ~1059 parity bits (out of 2208 available).
[NR-LDPC-STD] Requesting 1059 parity bits (Punctured Transmission).
[NR-LDPC-STD] Decoding Success! Parity checks passed.

[*] NR-LDPC decoding completed. Verifying results...

--- NR-LDPC Summary ---
[*] Initial Errors: 117
[*] Errors Remaining: 0
[*] Parity Bits Revealed (Leakage): 1059

### 5. LDPC Rate Adaptive Demo

In [12]:
async def demo_ldpc_rate_adaptive():
    N, qber = 1000, 0.1
    print(f"\n{'='*50}\n--- 5. LDPC Rate Adaptive Error Correction Demo ---\n{'='*50}")
    
    if LDPC_RateAdaptive_ClientProtocol is None:
        print("[-] LDPC Rate Adaptive is NOT available (TensorFlow/Sionna not installed).")
        print("[*] This protocol requires: pip install tensorflow sionna")
        print("[*] Skipping LDPC Rate Adaptive demo.\n")
        return None, None
    
    print(f"[*] Alice generates {N} bits of reference key.")
    print(f"[*] Bob receives key over noisy channel with ~{qber:.1%} probability of error per bit.")
    alice_key, bob_key = generate_keys(N, qber)
    initial_errors = sum(1 for a, b in zip(alice_key, bob_key) if a != b)
    print(f"[*] Actual errors introduced: {initial_errors} ({(initial_errors/N):.2%} QBER)\n")
    
    print("[*] Setting up Local Oracle (Alice) and Rate Adaptive LDPC Client (Bob)...")
    try:
        ldpc_ra = LDPC_RateAdaptive_ClientProtocol(verbose=True, r_max=0.333)
        print("[*] Bob executes Rate Adaptive decoding, estimating precise QBER and extracting punctured parity...")
        corrected, revealed, total_corrected, channel_uses = await ldpc_ra.run(bob_key, qber, LocalMockOracle(alice_key))
        
        print("\n[*] LDPC Rate Adaptive decoding completed. Verifying results...")
        final_errors = sum(1 for a, b in zip(alice_key, corrected) if a != b)
        print(f"\n--- LDPC Rate Adaptive Summary ---")
        print(f"[*] Initial Errors: {initial_errors}")
        print(f"[*] Errors Remaining: {final_errors}")
        print(f"[*] Parity Bits Revealed (Leakage): {revealed}")
        print(f"[*] Communication Rounds (Channel Uses): {channel_uses}")
        if final_errors == 0:
            print("[+] LDPC Rate Adaptive Match Success! Bob's key perfectly matches Alice's key.\n")
        else:
            print("[-] LDPC Rate Adaptive Match Failed! Some errors remain.\n")
        return alice_key, corrected
    except Exception as e:
        print(f"\n[-] LDPC Rate Adaptive Failed during execution: {e}\n")
        return None, None

ldpc_ra_keys = await demo_ldpc_rate_adaptive()


--- 5. LDPC Rate Adaptive Error Correction Demo ---
[*] Alice generates 1000 bits of reference key.
[*] Bob receives key over noisy channel with ~10.0% probability of error per bit.
[*] Actual errors introduced: 121 (12.10% QBER)

[*] Setting up Local Oracle (Alice) and Rate Adaptive LDPC Client (Bob)...
[*] Bob executes Rate Adaptive decoding, estimating precise QBER and extracting punctured parity...
[LDPC-RATE-ADAPTIVE] Starting Rate-Adaptive LDPC Protocol (K=1000, N_mother=3003)
[LDPC-RATE-ADAPTIVE] Syndrome-Based Error Est: 0.2000, Refined QBER=0.1900 (Prior=0.1000)
[LDPC-RATE-ADAPTIVE] Rate Adaptation: Target R_opt=0.158, Actual Code Rate=0.1584, Shortened=623, Punctured=0
[LDPC-RATE-ADAPTIVE] Computing LDPC Generator Matrix G (k=1000, n=3000)...
[LDPC-RATE-ADAPTIVE] Rate Adaptive Extraction: Firing 1052 LDPC Parity blocks...
[LDPC-RATE-ADAPTIVE] LDPC Belief Propagation complete. Corrected 121 bit flips.
[LDPC-RATE-ADAPTIVE] Confirmation Step: 4/4 subblocks verified identical.



### 6. Privacy Amplification Demo

In [10]:
def demo_privacy_amplification(alice_key, bob_key):
    print(f"\n{'='*50}\n--- 6. Privacy Amplification Demo ---\n{'='*50}")
    
    if alice_key is None or bob_key is None:
        print("[-] Privacy Amplification requires reconciled keys from error correction.")
        print("[*] Skipping Privacy Amplification demo (no valid reconciled key available).\n")
        return
    
    print("[*] Starting Privacy Amplification to compress partially exposed keys into a fully secure shorter key.")
    print("[*] Assumption: Alice and Bob have successfully reconciled their keys through Error Correction.\n")
    
    shared_seed = random.randint(0, 2**32 - 1)
    print(f"[*] Public random shared seed distributed: {shared_seed}")
    
    print("\n[*] Bob processing his key...")
    pa_bob = PrivacyAmplification(bob_key, qber=0.04, security_parameter=10)
    bob_secret = pa_bob.apply_hash(seed=shared_seed, algorithm='toeplitz')
    
    print("\n[*] Alice processing her key...")
    pa_alice = PrivacyAmplification(alice_key, qber=0.04, security_parameter=10)
    alice_secret = pa_alice.apply_hash(seed=shared_seed, algorithm='toeplitz')

    print(f"\n--- Privacy Amplification Summary ---")
    print(f"[*] Final Secrets Match: {alice_secret == bob_secret}")
    print(f"[*] Initial Reconciled Key Length: {len(alice_key)}")
    print(f"[*] Final Secure Secret Key Length: {len(bob_secret)}")
    if alice_secret == bob_secret:
        print("[+] Privacy Amplification Success! Both parties now share a secure final secret key.\n")
    else:
        print("[-] Privacy Amplification Warning: Secrets do not match!\n")

# Use matched_key from Cascade demo (which should succeed)
demo_privacy_amplification(alice_key, matched_key)


--- 6. Privacy Amplification Demo ---
[*] Starting Privacy Amplification to compress partially exposed keys into a fully secure shorter key.
[*] Assumption: Alice and Bob have successfully reconciled their keys through Error Correction.

[*] Public random shared seed distributed: 2985706840

[*] Bob processing his key...

[*] Alice processing her key...

--- Privacy Amplification Summary ---
[*] Final Secrets Match: True
[*] Initial Reconciled Key Length: 1000
[*] Final Secure Secret Key Length: 232
[+] Privacy Amplification Success! Both parties now share a secure final secret key.

